# Jalon 1 — Exploration de GTSRB et prétraitement CLAHE

**Responsable : Membre A**

Prérequis : `python scripts/download_data.py`

In [ ]:
import sys; sys.path.insert(0, '..')
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from clocker import config
from clocker.data import read_split
from clocker.labels import class_name
from clocker.preprocessing import load_rgb, crop_roi, apply_clahe

train_df = read_split('train'); test_df = read_split('test')
print(len(train_df), 'images train /', len(test_df), 'images test')
train_df.head()

## 1. Distribution des classes
Le dataset est déséquilibré : à mentionner dans le rapport et à compenser (class_weight, augmentation).

In [ ]:
counts = train_df['ClassId'].value_counts().sort_index()
fig, ax = plt.subplots(figsize=(14, 4))
ax.bar(counts.index, counts.values)
ax.set_xlabel('Classe'); ax.set_ylabel("Nombre d'images"); ax.set_xticks(range(43))
ax.set_title('Distribution des classes (train)')
plt.savefig('../reports/figures/class_distribution.png', dpi=150); plt.show()
print('min :', counts.min(), f'({class_name(counts.idxmin())})', '| max :', counts.max(), f'({class_name(counts.idxmax())})')

## 2. Tailles des images

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
axes[0].hist(train_df['Width'], bins=50); axes[0].set_title('Largeur (px)')
axes[1].hist(train_df['Height'], bins=50); axes[1].set_title('Hauteur (px)')
plt.show()
train_df[['Width', 'Height']].describe()

## 3. Un exemple par classe

In [ ]:
fig, axes = plt.subplots(5, 9, figsize=(16, 10))
for ax in axes.ravel(): ax.axis('off')
for cid, ax in zip(range(43), axes.ravel()):
    rows = train_df[train_df.ClassId == cid]
    row = rows.iloc[len(rows) // 2]
    ax.imshow(load_rgb(row.abs_path)); ax.set_title(str(cid), fontsize=9)
plt.tight_layout(); plt.savefig('../reports/figures/classes_examples.png', dpi=150); plt.show()

## 4. Effet de CLAHE (canal L, espace Lab)
On choisit les images les plus sombres pour bien voir l'effet, et on compare plusieurs `clipLimit`.

In [ ]:
sample = train_df.sample(2000, random_state=0).copy()
sample['brightness'] = [load_rgb(p).mean() for p in sample['abs_path']]
dark = sample.nsmallest(6, 'brightness')

clips = [None, 1.0, 2.0, 4.0]
fig, axes = plt.subplots(len(dark), len(clips), figsize=(9, 13))
for r, row in enumerate(dark.itertuples()):
    img = crop_roi(load_rgb(row.abs_path), row.x1, row.y1, row.x2, row.y2)
    for c, clip in enumerate(clips):
        out = img if clip is None else apply_clahe(img, clip_limit=clip)
        axes[r, c].imshow(out); axes[r, c].axis('off')
        if r == 0: axes[r, c].set_title('original' if clip is None else f'clip={clip}')
plt.tight_layout(); plt.savefig('../reports/figures/clahe_comparison.png', dpi=150); plt.show()

## 5. Construction du cache prétraité
Recadrage ROI → 32×32 → CLAHE → [0, 1], découpage train/val stratifié (85/15).

In [ ]:
from clocker.data import build_cache
build_cache(size=32, use_clahe=True)
build_cache(size=32, use_clahe=False)  # pour l'ablation sans CLAHE

## Observations à reporter dans le rapport
- …
- …